In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 7.9 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
google_gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051

In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://customize-watch-lifting.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://customize-watch-lifting.ngrok-free.dev


True

In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)

from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    print("EVENT: ", event)
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        line_bot_api.reply_message_with_http_info(
            ReplyMessageRequest(
                reply_token=event.reply_token,
                messages=[TextMessage(text=event.message.text),#在這邊會回應兩次
                            TextMessage(text=event.message.text)]
            )
        )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"Ub9ff2c3714925949563d00e62be17749","events":[{"type":"message","message":{"type":"text","id":"615026024115339394","quoteToken":"Mcy8HeeSjn3bP_MuPqA8kY9o5jF1xBKhD_I70JbIrk4gRm-H9KECqj50cFfYwCHeZstUEWX9Jdd23PacEA2g8_JdqVQYlF7-11lOAMOS6f6yL8JosiJTS_lvdX3l9vXBX6ilv_WnxK9aRHD_Q231bQ","markAsReadToken":"SsBY9m96gBlVjr_28s3lMqhDXm7Qy9fiWvtukaVReOTDMMLxvw5uUMilCp1X7qQiyPfPiw3_1ePx8Nd6ZF6neY3BFwocFJIGOTmzbSzN-ahqZxjJDyYCW7I-2CIeP-UBLueImDxWwg9_dkik2DJ-U6Od5v35Xwv5ekvr2TTMKDVWhZnGSSq4DlwhMB_1aY5GGqK1WO3J4s_-tbRB_2n4lA","text":"我是b12090023 陳俊安"},"webhookEventId":"01KS6PQXB9NKTVWF4FMV2RT7W6","deliveryContext":{"isRedelivery":false},"timestamp":1779415381091,"source":{"type":"user","userId":"U913e7b32d982838f15ac288880e33797"},"replyToken":"14629d18fb784a248cefd9d5968c69a6","mode":"active"}]}
EVENT:  type='message' source=UserSource(type='user', user_id='U913e7b32d982838f15ac288880e33797') timestamp=1779415381091 mode=<EventMode.ACTIVE: 'active'> webhook_event_id='01KS6PQXB9N

INFO:werkzeug:127.0.0.1 - - [22/May/2026 02:03:01] "POST / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [22/May/2026 02:03:14] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"Ub9ff2c3714925949563d00e62be17749","events":[{"type":"message","message":{"type":"text","id":"615026045674061932","quoteToken":"A-txhohCs1_XGlGN9urtj2_19cyKllUeCm-iiLETVJjcH_9vdb-rrhOe42PVJ5cum9SXVBQSpcFV7sCZpXyBNDTa5CRLnwFQ5NoO5F_yatl53ixi5SBzp_HMpA1e5p4CfHHg9ylf0cJyaXKYY6iHgw","markAsReadToken":"azw16WIwDFQe54h2YCrx1Of_d5bW2dXNKhOFfErXZa-OFnHCZZ3ZCL-dC9Lpp9ubm_QCOMu3XTb8BxSl3pdWmR5JTaj0FnSJdS973Gc4DNi5ctYPmyvnmjBBcurpkifpdAL-_JtMTtq1Btexsfa0CNLu3MbZ28LHXhmEUe2kI7WTy5NHxFROr1kiN_RFcaKf3xai_4hNSXsgg3Hvazf2Fw","text":"1"},"webhookEventId":"01KS6PR9N28WBXF33YG2N279SP","deliveryContext":{"isRedelivery":false},"timestamp":1779415393910,"source":{"type":"user","userId":"U913e7b32d982838f15ac288880e33797"},"replyToken":"decf2f3e4be74ee79406d6863e1dca34","mode":"active"}]}
EVENT:  type='message' source=UserSource(type='user', user_id='U913e7b32d982838f15ac288880e33797') timestamp=1779415393910 mode=<EventMode.ACTIVE: 'active'> webhook_event_id='01KS6PR9N28WBXF33YG2N279S